# 3단계 V5-3. 보이스피싱 특징 벡터·유사도 탐지

정상 금융상담을 사용하지 않고, 기존 보이스피싱 사례의 문구·행동·심리 특징을 수치화하여 신규 텍스트와의 유사도를 계산합니다.

> 보이스피싱 사례 정제 → 특징 벡터화 → 기준 라이브러리·전술 군집 구축 → 원본 파일 홀드아웃 검증 → 신규 텍스트 유사도·최근접 사례 출력

## 결과의 의미

- 높은 점수: 기존 보이스피싱 문구·전술과 유사함
- 중간 점수: 일부 특징이 유사함
- 낮은 점수: 기존 패턴과 유사성이 낮거나 입력 정보가 부족함

보이스피싱 사례만으로는 낮은 점수를 `정상`이라고 확정할 수 없습니다. 정상·비정상 임계값과 오탐률은 향후 별도의 정상·외부 데이터로 검증해야 합니다.


In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas pyarrow scikit-learn seaborn matplotlib koreanize-matplotlib joblib openpyxl


In [ ]:
# 1. 라이브러리와 Google Drive
from google.colab import drive
from pathlib import Path
from IPython.display import display
import hashlib, json, re, unicodedata, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
from scipy.sparse import hstack
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import normalize

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',100)
drive.mount('/content/drive')
print('Google Drive 연결 완료')


## 1. 경로와 분석 설정


In [ ]:
# 2. 입력·출력 경로
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
DATASET_ROOT = PROJECT_ROOT / '구축 데이터셋_v4'
STANDARD_ROOT = DATASET_ROOT / '01_standard_tables'
ML_ROOT = DATASET_ROOT / '02_ml_tables'
OUTPUT_ROOT = PROJECT_ROOT / '머신러닝_분석결과_v5_3_유사도탐지'
TABLE_ROOT = OUTPUT_ROOT / '01_특징점수표'
FIGURE_ROOT = OUTPUT_ROOT / '02_그래프'
MODEL_ROOT = OUTPUT_ROOT / '03_유사도모델'
TEST_ROOT = OUTPUT_ROOT / '04_홀드아웃검증'
REPORT_ROOT = OUTPUT_ROOT / '05_보고서'
for folder in [TABLE_ROOT,FIGURE_ROOT,MODEL_ROOT,TEST_ROOT,REPORT_ROOT]:
    folder.mkdir(parents=True,exist_ok=True)

SEED = 42
TEST_RATIO = 0.20
MIN_TEXT_LENGTH = 20
MAX_FEATURES = 60000
WORD_WEIGHT = 0.60
TACTIC_WEIGHT = 0.25
CLUSTER_WEIGHT = 0.15
LOW_QUANTILE = 0.10
HIGH_QUANTILE = 0.50
assert abs(WORD_WEIGHT+TACTIC_WEIGHT+CLUSTER_WEIGHT-1.0) < 1e-9
assert STANDARD_ROOT.exists() and ML_ROOT.exists(),f'구축 데이터셋_v4 경로를 확인하세요: {DATASET_ROOT}'
print('입력:',DATASET_ROOT); print('출력:',OUTPUT_ROOT)


## 2. 보이스피싱 기준 사례 준비


In [ ]:
# 3. 정상상담을 불러오지 않고 보이스피싱 사건 테이블만 사용합니다.
def read_table(folder,name):
    pq=folder/f'{name}.parquet'; csv=folder/f'{name}.csv'
    if pq.exists(): return pd.read_parquet(pq)
    assert csv.exists(),f'{name}을 찾지 못했습니다: {folder}'
    return pd.read_csv(csv,encoding='utf-8-sig')

case_df=read_table(STANDARD_ROOT,'vp_cases')
cluster_df=read_table(ML_ROOT,'case_clustering_ml')
assert {'case_id','file_id','model_input_text'}.issubset(cluster_df.columns)
print('사건:',len(case_df),'건 / 유사도 후보:',len(cluster_df),'건')
print('정상 금융상담 사용: 0건')


In [ ]:
# 4. 원문은 보존하고 모델용 텍스트만 정리합니다.
def clean_text(value):
    text=unicodedata.normalize('NFKC',str(value or '')).lower()
    text=re.sub(r'https?://\S+|www\.\S+',' ',text)
    text=re.sub(r'(?m)^\s*(tx|rx|화자\s*\d*|범인|피해자)\s*[:：]\s*',' ',text)
    text=re.sub(r'[*#xX]{2,}',' 마스킹 ',text)
    text=re.sub(r'\b\d{2,}\b',' 숫자 ',text)
    return re.sub(r'\s+',' ',text).strip()

reference=cluster_df.copy()
reference['clean_text']=reference['model_input_text'].fillna('').map(clean_text)
reference=reference[reference.clean_text.str.len()>=MIN_TEXT_LENGTH].copy()
reference['text_hash']=reference.clean_text.map(lambda x:hashlib.sha256(x.encode()).hexdigest())
reference=reference.drop_duplicates(['file_id','text_hash']).reset_index(drop=True)
reference['group_id']=reference['file_id'].astype(str)
reference['text_length']=reference.clean_text.str.len()
assert len(reference)>50 and reference.group_id.notna().all()
display(reference[['case_id','file_id','source_category','text_length']].head())
print('최종 보이스피싱 기준 후보:',len(reference),'건')


## 3. 행동·심리·말투 특징 수치화


In [ ]:
# 5. 연구자가 정한 특징 사전: 자동 추출 결과는 SILVER 특징입니다.
FEATURE_RULES={
 'money_request':r'송금|이체|입금|납부|지불|결제|돈.{0,8}(보내|내|줘|준비)|금액.{0,8}(보내|입금)',
 'transfer_cash':r'계좌.{0,8}(이체|송금)|현금.{0,8}(인출|찾|전달)|ATM|씨디기|CD기',
 'fee_tax_deposit':r'수수료|선입금|보증금|예치금|공탁금|세금|과태료|벌금|인지대',
 'account_info':r'계좌번호|잔액|통장|카드번호|금융거래|거래내역',
 'personal_info':r'주민번호|주민등록|생년월일|신분증|주소|개인정보|명의',
 'auth_code':r'인증번호|비밀번호|보안카드|OTP|일회용.{0,3}비밀번호',
 'app_remote':r'앱.{0,8}(설치|깔)|어플.{0,8}(설치|깔)|원격.{0,8}(접속|제어)|팀뷰어|퀵서포트',
 'command_pressure':r'하세요|하셔야|해야 합니다|따라 하|지금.{0,8}(가|하|보내|이체)|시키는 대로|말씀드린 대로',
 'urgency_pressure':r'지금 당장|즉시|긴급|오늘 안|시간이 없|빨리|지체하면|마감|몇 분 안',
 'fear_threat':r'체포|구속|압류|범죄|수배|처벌|고소|고발|피해를 입|큰일|위험|납치',
 'isolation_secrecy':r'비밀|말하지 마|알리면 안|누구에게도|혼자만|통화.{0,8}(끊지|유지)|전화.{0,8}(끊지|받지)',
 'authority_trust':r'검찰|검사|경찰|수사관|법원|금융감독원|금감원|은행 본점|정부기관|공문|사건번호',
 'resistance_handling':r'의심|못 믿|확인해 보|그게 아니라|걱정하지|안심|오해|설명드리',
 'benefit_offer':r'대출.{0,10}(승인|가능|해드리)|환급|돌려드리|지원금|혜택|저금리|금리.{0,8}(낮|인하)|한도.{0,8}(상향|증액)',
}
FEATURE_KO={
 'money_request':'금전·송금요구','transfer_cash':'송금·현금행동','fee_tax_deposit':'수수료·세금·보증금',
 'account_info':'계좌정보','personal_info':'개인정보','auth_code':'인증정보','app_remote':'앱설치·원격접속',
 'command_pressure':'명령·강압','urgency_pressure':'긴급성·시간압박','fear_threat':'공포·위협',
 'isolation_secrecy':'고립·비밀유지','authority_trust':'권위·신뢰형성',
 'resistance_handling':'의심·저항대응','benefit_offer':'이익·혜택제안'}
COMPILED_RULES={name:re.compile(pattern,re.I) for name,pattern in FEATURE_RULES.items()}

def extract_tactic_features(text):
    denominator=max(len(text),1); row={}; present=0
    for name,pattern in COMPILED_RULES.items():
        count=len(pattern.findall(text))
        row[f'{name}_rate_1k']=count/denominator*1000
        row[f'{name}_flag']=int(count>0)
        present+=int(count>0)
    row['risk_feature_diversity_ratio']=present/len(COMPILED_RULES)
    return row

tactic_values=pd.DataFrame(reference.clean_text.map(extract_tactic_features).tolist(),index=reference.index)
tactic_columns=list(tactic_values.columns)
reference_features=pd.concat([reference[['case_id','file_id','source_category','text_length']],tactic_values],axis=1)
reference_features.to_csv(TABLE_ROOT/'보이스피싱_사건별_특징벡터.csv',index=False,encoding='utf-8-sig')
display(reference_features.head()); print('수치화 특징:',len(tactic_columns),'개')


## 4. 평가용 원본 파일 홀드아웃


In [ ]:
# 6. 같은 원본 파일의 사건이 기준 라이브러리와 평가에 동시에 들어가지 않게 분리합니다.
groups=np.array(sorted(reference.group_id.unique()))
reference_groups,holdout_groups=train_test_split(groups,test_size=TEST_RATIO,random_state=SEED)
train_ref=reference[reference.group_id.isin(reference_groups)].reset_index(drop=True)
holdout=reference[reference.group_id.isin(holdout_groups)].reset_index(drop=True)
assert set(train_ref.group_id).isdisjoint(set(holdout.group_id))
split_summary=pd.DataFrame([
    {'구분':'REFERENCE','원본파일수':train_ref.group_id.nunique(),'사건수':len(train_ref)},
    {'구분':'HOLDOUT_FRAUD','원본파일수':holdout.group_id.nunique(),'사건수':len(holdout)},
])
display(split_summary)
split_summary.to_csv(TEST_ROOT/'홀드아웃_분리요약.csv',index=False,encoding='utf-8-sig')


## 5. 문구·전술·군집 기준 라이브러리 구축


In [ ]:
# 7. 단어와 문자 TF-IDF를 함께 사용합니다.
def make_text_vectorizer():
    return FeatureUnion([
      ('word',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=30000,sublinear_tf=True)),
      ('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=30000,sublinear_tf=True)),
    ])

text_vectorizer=make_text_vectorizer()
train_text_x=text_vectorizer.fit_transform(train_ref.clean_text)
holdout_text_x=text_vectorizer.transform(holdout.clean_text)
text_nn=NearestNeighbors(metric='cosine',algorithm='brute',n_neighbors=min(50,len(train_ref)))
text_nn.fit(train_text_x)
print('문구 특징 행렬:',train_text_x.shape)


In [ ]:
# 8. 행동·심리 특징과 텍스트 축약 벡터를 결합해 전술 군집을 만듭니다.
def tactic_matrix(texts):
    return pd.DataFrame(pd.Series(texts).map(extract_tactic_features).tolist())[tactic_columns].to_numpy(float)

train_tactic_raw=tactic_matrix(train_ref.clean_text)
holdout_tactic_raw=tactic_matrix(holdout.clean_text)
train_tactic_x=normalize(train_tactic_raw,norm='l2')
holdout_tactic_x=normalize(holdout_tactic_raw,norm='l2')

n_components=min(100,train_text_x.shape[0]-1,train_text_x.shape[1]-1)
assert n_components>=2,'SVD에 필요한 사건 또는 특징이 부족합니다.'
svd=TruncatedSVD(n_components=n_components,random_state=SEED)
train_svd=normalize(svd.fit_transform(train_text_x),norm='l2')
holdout_svd=normalize(svd.transform(holdout_text_x),norm='l2')
train_cluster_x=np.hstack([train_svd,train_tactic_x])
holdout_cluster_x=np.hstack([holdout_svd,holdout_tactic_x])

k_rows=[]; k_models={}
for k in range(2,9):
    model=KMeans(n_clusters=k,n_init=30,random_state=SEED)
    labels=model.fit_predict(train_cluster_x)
    other=KMeans(n_clusters=k,n_init=30,random_state=SEED+1).fit_predict(train_cluster_x)
    k_rows.append({'군집수':k,'silhouette':silhouette_score(train_cluster_x,labels),
                   'seed_stability_ari':adjusted_rand_score(labels,other)})
    k_models[k]=(model,labels)
k_compare=pd.DataFrame(k_rows).sort_values(['silhouette','seed_stability_ari'],ascending=False)
best_k=int(k_compare.iloc[0]['군집수'])
kmeans,train_clusters=k_models[best_k]
train_ref['전술군집ID']=train_clusters
display(k_compare)
k_compare.to_csv(TABLE_ROOT/'전술군집_K비교.csv',index=False,encoding='utf-8-sig')


In [ ]:
# 9. 유사도 계산 공통 함수
def choose_neighbor(distances,indices,query_groups=None,exclude_same_group=False):
    chosen_similarity=[]; chosen_index=[]
    reference_group_array=train_ref.group_id.to_numpy()
    for row_no,(row_dist,row_idx) in enumerate(zip(distances,indices)):
        selected=0
        if exclude_same_group and query_groups is not None:
            valid=np.where(reference_group_array[row_idx]!=str(query_groups[row_no]))[0]
            if len(valid): selected=int(valid[0])
        chosen_similarity.append(float(np.clip(1-row_dist[selected],0,1)))
        chosen_index.append(int(row_idx[selected]))
    return np.array(chosen_similarity),np.array(chosen_index)

def score_vectors(text_x,tactic_x,cluster_x,query_groups=None,exclude_same_group=False):
    distances,indices=text_nn.kneighbors(text_x)
    text_score,nearest_index=choose_neighbor(distances,indices,query_groups,exclude_same_group)
    tactic_similarity=np.asarray(tactic_x@train_tactic_x.T)
    if exclude_same_group and query_groups is not None:
        for i,g in enumerate(query_groups): tactic_similarity[i,train_ref.group_id.to_numpy()==str(g)]=-1
    tactic_score=np.clip(tactic_similarity.max(axis=1),0,1)
    centers=normalize(kmeans.cluster_centers_,norm='l2')
    cluster_score=np.clip(np.asarray(normalize(cluster_x,norm='l2')@centers.T).max(axis=1),0,1)
    combined=WORD_WEIGHT*text_score+TACTIC_WEIGHT*tactic_score+CLUSTER_WEIGHT*cluster_score
    cluster_id=kmeans.predict(cluster_x)
    return pd.DataFrame({'문구유사도':text_score,'전술유사도':tactic_score,
        '군집유사도':cluster_score,'종합유사도':combined,'전술군집ID':cluster_id,
        '최근접기준행':nearest_index})


## 6. 보이스피싱 내부 기준점과 홀드아웃 검증


In [ ]:
# 10. REFERENCE 사건은 자기 자신과 같은 원본 파일을 제외하고 점수화합니다.
train_dist,train_idx=text_nn.kneighbors(train_text_x)
train_text_score,train_nearest=choose_neighbor(
    train_dist,train_idx,train_ref.group_id.to_numpy(),exclude_same_group=True
)
train_tactic_similarity=np.asarray(train_tactic_x@train_tactic_x.T)
same_group=train_ref.group_id.to_numpy()[:,None]==train_ref.group_id.to_numpy()[None,:]
train_tactic_similarity[same_group]=-1
train_tactic_score=np.clip(train_tactic_similarity.max(axis=1),0,1)
centers=normalize(kmeans.cluster_centers_,norm='l2')
train_cluster_score=np.clip(np.asarray(normalize(train_cluster_x,norm='l2')@centers.T).max(axis=1),0,1)
train_combined=WORD_WEIGHT*train_text_score+TACTIC_WEIGHT*train_tactic_score+CLUSTER_WEIGHT*train_cluster_score

LOW_THRESHOLD=float(np.quantile(train_combined,LOW_QUANTILE))
HIGH_THRESHOLD=float(np.quantile(train_combined,HIGH_QUANTILE))
threshold_df=pd.DataFrame([
 {'구간':'기존패턴과 낮은 유사도·검토','최소점수':0.0,'최대점수':LOW_THRESHOLD},
 {'구간':'기존 보이스피싱과 유사','최소점수':LOW_THRESHOLD,'최대점수':HIGH_THRESHOLD},
 {'구간':'기존 보이스피싱과 매우 유사','최소점수':HIGH_THRESHOLD,'최대점수':1.0},
])
display(threshold_df)
threshold_df.to_csv(TABLE_ROOT/'유사도_임시구간.csv',index=False,encoding='utf-8-sig')
print('주의: 이 기준은 보이스피싱 내부 분위수이며 정상/비정상 판정 임계값이 아닙니다.')


In [ ]:
# 11. 보지 않은 원본 파일의 보이스피싱이 기준 라이브러리와 얼마나 유사한지 측정합니다.
holdout_score=score_vectors(holdout_text_x,holdout_tactic_x,holdout_cluster_x)
holdout_result=pd.concat([holdout[['case_id','file_id','source_category','clean_text']],holdout_score],axis=1)
holdout_result['유사도구간']=np.select(
    [holdout_result.종합유사도>=HIGH_THRESHOLD,holdout_result.종합유사도>=LOW_THRESHOLD],
    ['기존 보이스피싱과 매우 유사','기존 보이스피싱과 유사'],
    default='기존패턴과 낮은 유사도·검토')
holdout_result['최근접_case_id']=holdout_result['최근접기준행'].map(train_ref.case_id)
known_fraud_coverage=float((holdout_result.종합유사도>=LOW_THRESHOLD).mean())
holdout_metrics=pd.DataFrame([{
    '홀드아웃_보이스피싱수':len(holdout_result),
    '종합유사도_평균':holdout_result.종합유사도.mean(),
    '종합유사도_중앙값':holdout_result.종합유사도.median(),
    '임시하한이상_포착률':known_fraud_coverage,
    '정상데이터수':0,'정밀도_특이도_오탐률':'계산불가'
}])
display(holdout_metrics); display(holdout_result.head())
holdout_metrics.to_csv(TEST_ROOT/'홀드아웃_보이스피싱_포착결과.csv',index=False,encoding='utf-8-sig')
holdout_result.to_csv(TEST_ROOT/'홀드아웃_사건별_유사도.csv',index=False,encoding='utf-8-sig')

plt.figure(figsize=(10,5)); sns.histplot(holdout_result.종합유사도,bins=25,kde=True)
plt.axvline(LOW_THRESHOLD,color='orange',ls='--',label='임시 하한')
plt.axvline(HIGH_THRESHOLD,color='red',ls='--',label='높은 유사도 기준')
plt.xlabel('종합 유사도'); plt.title('홀드아웃 보이스피싱 유사도 분포'); plt.legend(); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'홀드아웃_보이스피싱_유사도분포.png',dpi=170); plt.show()


## 7. 전체 사건으로 최종 기준 라이브러리 재구축

홀드아웃 검증이 끝난 후 서비스 실험용 기준 라이브러리는 전체 보이스피싱 사건으로 다시 구축합니다.


In [ ]:
# 12. 전체 사건으로 최종 문구·전술·군집 모델 재학습
final_ref=reference.reset_index(drop=True).copy()
final_vectorizer=make_text_vectorizer()
final_text_x=final_vectorizer.fit_transform(final_ref.clean_text)
final_nn=NearestNeighbors(metric='cosine',algorithm='brute',n_neighbors=min(50,len(final_ref))).fit(final_text_x)
final_tactic_raw=tactic_matrix(final_ref.clean_text)
final_tactic_x=normalize(final_tactic_raw,norm='l2')
final_components=min(100,final_text_x.shape[0]-1,final_text_x.shape[1]-1)
final_svd=TruncatedSVD(n_components=final_components,random_state=SEED)
final_svd_x=normalize(final_svd.fit_transform(final_text_x),norm='l2')
final_cluster_x=np.hstack([final_svd_x,final_tactic_x])
final_kmeans=KMeans(n_clusters=best_k,n_init=30,random_state=SEED).fit(final_cluster_x)
final_ref['전술군집ID']=final_kmeans.labels_
final_ref[['case_id','file_id','source_category','전술군집ID','clean_text']].to_csv(
    TABLE_ROOT/'최종_보이스피싱_기준라이브러리.csv',index=False,encoding='utf-8-sig')
print('최종 기준 사례:',len(final_ref),'건 / 군집:',best_k,'개')


In [ ]:
# 13. 전체 기준 라이브러리용 신규 텍스트 채점 함수
def similarity_level(score,text_length):
    if text_length<MIN_TEXT_LENGTH: return '입력정보부족'
    if score>=HIGH_THRESHOLD: return '기존 보이스피싱과 매우 유사'
    if score>=LOW_THRESHOLD: return '기존 보이스피싱과 유사'
    return '기존패턴과 낮은 유사도·추가검토'

def score_new_texts(texts,top_k=3):
    clean=[clean_text(x) for x in texts]
    text_x=final_vectorizer.transform(clean)
    tactic_raw=tactic_matrix(clean)
    tactic_x=normalize(tactic_raw,norm='l2')
    svd_x=normalize(final_svd.transform(text_x),norm='l2')
    cluster_x=np.hstack([svd_x,tactic_x])

    distances,indices=final_nn.kneighbors(text_x,n_neighbors=min(max(top_k,1),len(final_ref)))
    text_score=np.clip(1-distances[:,0],0,1)
    tactic_similarity=np.asarray(tactic_x@final_tactic_x.T)
    tactic_score=np.clip(tactic_similarity.max(axis=1),0,1)
    centers=normalize(final_kmeans.cluster_centers_,norm='l2')
    cluster_score=np.clip(np.asarray(normalize(cluster_x,norm='l2')@centers.T).max(axis=1),0,1)
    combined=WORD_WEIGHT*text_score+TACTIC_WEIGHT*tactic_score+CLUSTER_WEIGHT*cluster_score
    cluster_id=final_kmeans.predict(cluster_x)

    rows=[]; neighbor_rows=[]
    for i,text in enumerate(clean):
        rows.append({'입력번호':i+1,'입력텍스트':text,'문자수':len(text),'문구유사도':text_score[i],
          '전술유사도':tactic_score[i],'군집유사도':cluster_score[i],'종합유사도':combined[i],
          '전술군집ID':int(cluster_id[i]),'판정':similarity_level(combined[i],len(text))})
        for rank,(distance,idx) in enumerate(zip(distances[i],indices[i]),1):
            item=final_ref.iloc[int(idx)]
            neighbor_rows.append({'입력번호':i+1,'유사순위':rank,'유사도':float(np.clip(1-distance,0,1)),
              '기준_case_id':item.case_id,'원본분류':item.source_category,'기준문장':item.clean_text[:500]})
    return pd.DataFrame(rows),pd.DataFrame(neighbor_rows)

print('score_new_texts 함수 준비 완료')


## 8. 신규 텍스트 입력 및 유사도 확인

아래 `NEW_TEXTS` 목록에 분석할 텍스트를 넣습니다. 여러 문장을 한 번에 입력할 수 있습니다.


In [ ]:
# 14. 사용자가 분석할 신규 텍스트
NEW_TEXTS = [
    # "여기에 확인할 통화 전사 텍스트를 입력하세요",
]

if NEW_TEXTS:
    new_scores,new_neighbors=score_new_texts(NEW_TEXTS,top_k=3)
    display(new_scores); display(new_neighbors)
    new_scores.to_csv(TABLE_ROOT/'신규텍스트_유사도점수.csv',index=False,encoding='utf-8-sig')
    new_neighbors.to_csv(TABLE_ROOT/'신규텍스트_유사사건_TOP3.csv',index=False,encoding='utf-8-sig')
else:
    print('NEW_TEXTS에 분석할 텍스트를 입력한 뒤 이 셀을 다시 실행하세요.')


## 9. 모델·기준표·보고서 저장


In [ ]:
# 15. 최종 유사도 패키지 저장
model_package={
 'version':'v5_3_similarity','text_vectorizer':final_vectorizer,'nearest_neighbors':final_nn,
 'svd':final_svd,'kmeans':final_kmeans,'reference_tactic_matrix':final_tactic_x,
 'reference_metadata':final_ref[['case_id','file_id','source_category','전술군집ID','clean_text']],
 'feature_rules':FEATURE_RULES,'tactic_columns':tactic_columns,
 'weights':{'text':WORD_WEIGHT,'tactic':TACTIC_WEIGHT,'cluster':CLUSTER_WEIGHT},
 'thresholds':{'low':LOW_THRESHOLD,'high':HIGH_THRESHOLD},
 'threshold_meaning':'fraud_internal_quantiles_not_normal_abnormal_boundary',
}
joblib.dump(model_package,MODEL_ROOT/'voicephishing_similarity_v5_3.joblib')

report=[
 '# V5-3 보이스피싱 특징·유사도 탐지 결과','', '## Summary','',
 f'- 최종 보이스피싱 기준 사례: {len(final_ref):,}건',
 f'- 전술 군집 수: {best_k}개',
 f'- 보이스피싱 내부 임시 유사도 하한: {LOW_THRESHOLD:.4f}',
 f'- 높은 유사도 기준: {HIGH_THRESHOLD:.4f}',
 f'- 홀드아웃 보이스피싱 임시 하한 이상 포착률: {known_fraud_coverage:.2%}','',
 '## 점수 구성','',
 f'- 문구 유사도 가중치: {WORD_WEIGHT:.0%}',f'- 전술 유사도 가중치: {TACTIC_WEIGHT:.0%}',
 f'- 군집 유사도 가중치: {CLUSTER_WEIGHT:.0%}','',
 '## 제한','',
 '- 정상 금융상담을 사용하지 않아 정상·비정상 분류 정확도, 정밀도, 특이도와 오탐률을 계산할 수 없습니다.',
 '- 낮은 유사도는 정상 확정이 아니라 기존 패턴과 다르거나 입력 정보가 부족하다는 뜻입니다.',
 '- 새로운 유형의 보이스피싱도 낮은 점수가 나올 수 있으므로 추가 검토 대상으로 처리합니다.',
 '- 행동·심리 특징은 연구자가 정한 규칙 기반 SILVER 특징입니다.',
 '- 최종 정상/의심 임계값은 정상·외부 보이스피싱 데이터로 별도 교정해야 합니다.'
]
(REPORT_ROOT/'03_v5_3_similarity_report.md').write_text('\n'.join(report),encoding='utf-8')
manifest={'version':'v5_3_similarity','dataset_root':str(DATASET_ROOT),'output_root':str(OUTPUT_ROOT),
 'normal_rows_used':0,'reference_cases':len(final_ref),'seed':SEED,'best_k':best_k,
 'low_threshold':LOW_THRESHOLD,'high_threshold':HIGH_THRESHOLD,
 'heldout_known_fraud_coverage':known_fraud_coverage,'group_leakage_check':True,
 'score_name':'보이스피싱 유사도','not_a_normal_abnormal_classifier':True}
(REPORT_ROOT/'03_v5_3_run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

# 저장된 함수와 모델이 실제로 점수를 반환하는지 기준 사건 1건으로 확인
smoke_score,smoke_neighbors=score_new_texts([final_ref.iloc[0].clean_text],top_k=1)
assert smoke_score.종합유사도.between(0,1).all()
assert len(smoke_neighbors)==1
assert set(reference_groups).isdisjoint(set(holdout_groups))
assert manifest['normal_rows_used']==0
print('V5-3 유사도 탐지 정상 완료:',OUTPUT_ROOT)
